In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 21


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.969215080142021
Epoch 2/100, Loss: 2.979239299893379
Epoch 3/100, Loss: 3.346597172319889
Epoch 4/100, Loss: 2.8511520102620125
Epoch 5/100, Loss: 2.801318295300007
Epoch 6/100, Loss: 3.033502459526062
Epoch 7/100, Loss: 2.8382134810090065
Epoch 8/100, Loss: 2.940922886133194
Epoch 9/100, Loss: 3.021699897944927
Epoch 10/100, Loss: 3.0578396916389465
Epoch 11/100, Loss: 2.933562181890011
Epoch 12/100, Loss: 2.966615989804268
Epoch 13/100, Loss: 2.8332561925053596
Epoch 14/100, Loss: 2.843479558825493
Epoch 15/100, Loss: 3.0867812782526016


Epoch 16/100, Loss: 2.8144714534282684
Epoch 17/100, Loss: 2.994497038424015
Epoch 18/100, Loss: 3.047878809273243
Epoch 19/100, Loss: 2.8665915429592133
Epoch 20/100, Loss: 2.998001664876938
Epoch 21/100, Loss: 2.9800333976745605
Epoch 22/100, Loss: 2.7912193536758423
Epoch 23/100, Loss: 2.9165675044059753
Epoch 24/100, Loss: 2.970160588622093
Epoch 25/100, Loss: 2.8817678838968277
Epoch 26/100, Loss: 3.064101003110409
Epoch 27/100, Loss: 2.910935804247856
Epoch 28/100, Loss: 2.8026096746325493
Epoch 29/100, Loss: 2.9648825004696846
Epoch 30/100, Loss: 2.831975869834423


Epoch 31/100, Loss: 2.809829965233803
Epoch 32/100, Loss: 2.838487893342972
Epoch 33/100, Loss: 2.996773637831211
Epoch 34/100, Loss: 2.917971931397915
Epoch 35/100, Loss: 2.8404457718133926
Epoch 36/100, Loss: 2.8253123238682747
Epoch 37/100, Loss: 2.9236972257494926
Epoch 38/100, Loss: 2.873792976140976
Epoch 39/100, Loss: 2.9939015805721283
Epoch 40/100, Loss: 2.9098857045173645
Epoch 41/100, Loss: 2.9169055595993996
Epoch 42/100, Loss: 3.0006302297115326
Epoch 43/100, Loss: 3.069574475288391
Epoch 44/100, Loss: 2.9688103199005127
Epoch 45/100, Loss: 2.878281347453594


Epoch 46/100, Loss: 2.9872818514704704
Epoch 47/100, Loss: 2.8145057260990143
Epoch 48/100, Loss: 2.8764853551983833
Epoch 49/100, Loss: 2.9768344908952713
Epoch 50/100, Loss: 2.8732775896787643
Epoch 51/100, Loss: 2.960905149579048
Epoch 52/100, Loss: 2.8757039308547974
Epoch 53/100, Loss: 2.9810776337981224
Epoch 54/100, Loss: 2.969970278441906
Epoch 55/100, Loss: 2.8636528626084328
Epoch 56/100, Loss: 2.953383058309555
Epoch 57/100, Loss: 2.931519739329815
Epoch 58/100, Loss: 2.8752604946494102
Epoch 59/100, Loss: 2.916870415210724
Epoch 60/100, Loss: 2.887612819671631


Epoch 61/100, Loss: 2.921439416706562
Epoch 62/100, Loss: 2.973442420363426
Epoch 63/100, Loss: 2.8788426965475082
Epoch 64/100, Loss: 2.948086567223072
Epoch 65/100, Loss: 2.9961299672722816
Epoch 66/100, Loss: 2.9145163744688034
Epoch 67/100, Loss: 2.8890255093574524
Epoch 68/100, Loss: 2.8547732532024384
Epoch 69/100, Loss: 2.8441299945116043
Epoch 70/100, Loss: 2.9658168181777
Epoch 71/100, Loss: 2.923827037215233
Epoch 72/100, Loss: 2.9408784359693527
Epoch 73/100, Loss: 2.890777640044689
Epoch 74/100, Loss: 3.0501460805535316
Epoch 75/100, Loss: 2.943427160382271
Epoch 76/100, Loss: 2.7968073189258575


...

Epoch 22/100, Loss: 1.67038843780756
Epoch 23/100, Loss: 1.6859115809202194
Epoch 24/100, Loss: 1.682744812220335
Epoch 25/100, Loss: 1.7778316587209702
Epoch 26/100, Loss: 1.7525271475315094
Epoch 27/100, Loss: 1.721642628312111
Epoch 28/100, Loss: 1.6990168988704681
Epoch 29/100, Loss: 1.588443148881197
Epoch 30/100, Loss: 1.6570327505469322
Epoch 31/100, Loss: 1.679975837469101
Epoch 32/100, Loss: 1.6613469570875168
Epoch 33/100, Loss: 1.7482316754758358
Epoch 34/100, Loss: 1.5957626774907112
Epoch 35/100, Loss: 1.6805567368865013
Epoch 36/100, Loss: 1.6102065369486809


Epoch 37/100, Loss: 1.575065754354
Epoch 38/100, Loss: 1.9485089033842087
Epoch 39/100, Loss: 1.7930903509259224
Epoch 40/100, Loss: 1.6064252331852913
Epoch 41/100, Loss: 1.8007291667163372
Epoch 42/100, Loss: 1.6972138807177544
Epoch 43/100, Loss: 1.7486656606197357
Epoch 44/100, Loss: 1.643850665539503
Epoch 45/100, Loss: 1.6358116194605827
Epoch 46/100, Loss: 1.7757186479866505
Epoch 47/100, Loss: 1.675596222281456
Epoch 48/100, Loss: 1.656347580254078
Epoch 49/100, Loss: 1.7064896896481514
Epoch 50/100, Loss: 1.6647423580288887
Epoch 51/100, Loss: 1.694336399435997
Epoch 52/100, Loss: 1.7733459658920765
Epoch 53/100, Loss: 1.6959257572889328
Epoch 54/100, Loss: 1.8050592839717865


Epoch 55/100, Loss: 1.7571628130972385
Epoch 56/100, Loss: 1.6552201583981514
Epoch 57/100, Loss: 1.634898267686367
Epoch 58/100, Loss: 1.7159782126545906
Epoch 59/100, Loss: 1.6302117928862572
Epoch 60/100, Loss: 1.7238698899745941
Epoch 61/100, Loss: 1.6474230736494064
Epoch 62/100, Loss: 1.9024660922586918
Epoch 63/100, Loss: 1.6449095159769058
Epoch 64/100, Loss: 1.581168346107006
Epoch 65/100, Loss: 1.67998818308115
Epoch 66/100, Loss: 1.7521563097834587
Epoch 67/100, Loss: 1.8177175223827362
Epoch 68/100, Loss: 1.6268198937177658
Epoch 69/100, Loss: 1.6648856848478317


Epoch 70/100, Loss: 1.6974961906671524
Epoch 71/100, Loss: 1.7900784835219383
Epoch 72/100, Loss: 1.72441878169775
Epoch 73/100, Loss: 1.6841182485222816
Epoch 74/100, Loss: 1.7019989639520645
Epoch 75/100, Loss: 1.6319009736180305
Epoch 76/100, Loss: 1.7308470904827118
Epoch 77/100, Loss: 1.6251276955008507
Epoch 78/100, Loss: 1.7918235212564468
Epoch 79/100, Loss: 1.7420797869563103
Epoch 80/100, Loss: 1.7175204455852509
Epoch 81/100, Loss: 1.7381179556250572
Epoch 82/100, Loss: 1.6607603281736374
Epoch 83/100, Loss: 1.7235995307564735
Epoch 84/100, Loss: 1.6933659091591835
Epoch 85/100, Loss: 1.7421742752194405


Epoch 86/100, Loss: 1.7186598144471645
Epoch 87/100, Loss: 1.7388661354780197
Epoch 88/100, Loss: 1.7082982063293457
Epoch 89/100, Loss: 1.6839435547590256
Epoch 90/100, Loss: 1.6804335415363312
Epoch 91/100, Loss: 1.8645017072558403
Epoch 92/100, Loss: 1.6129843071103096
Epoch 93/100, Loss: 1.7301015406847
Epoch 94/100, Loss: 1.7845942676067352
Epoch 95/100, Loss: 1.6230056956410408
Epoch 96/100, Loss: 1.6172618344426155
Epoch 97/100, Loss: 1.6400361955165863
Epoch 98/100, Loss: 1.7836412712931633
Epoch 99/100, Loss: 1.6813677288591862
Epoch 100/100, Loss: 1.796965777873993
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5711
